# Preditor de sucesso cinematográfico + LFM3 local (Ollama)

Notebook para Google Colab (GPU recomendada): instala Ollama, inicia o servidor no padrão do Ollama Colab Runner, baixa `lfm3:1.7b`, treina o preditor NumPy com a base TMDB e abre um chat que pergunta pelos dados faltantes.

> A primeira instalação e o download do modelo precisam de internet e podem demorar. O modelo ocupa aproximadamente 1,1 GB. No Colab, GPU é recomendada, mas a disponibilidade depende do tipo de sessão. O runtime pode reiniciar e apagar os arquivos baixados.

A previsão é estimativa histórica: sucesso significa bilheteria de pelo menos 2x o orçamento e não equivale a lucro.

## 1. Instalar dependências e Ollama

Execute as células na ordem. A instalação usa o instalador oficial do Ollama para Linux, ambiente usado pelo Colab.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq pciutils lshw zstd
!curl -fsSL https://ollama.com/install.sh | sh
%pip -q install numpy pandas openai ollama
import os, shutil, subprocess

assert shutil.which('ollama'), 'Instalação do Ollama não encontrou o executável.'
# O Colab já fornece o driver NVIDIA. Não instale cuda-drivers via apt.
nvidia_lib = '/usr/lib64-nvidia'
if os.path.isdir(nvidia_lib):
    os.environ['LD_LIBRARY_PATH'] = nvidia_lib + ':' + os.environ.get('LD_LIBRARY_PATH', '')
print('Ollama instalado:', shutil.which('ollama'))
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout[:1000] if gpu.returncode == 0 else 'GPU NVIDIA não detectada; Ollama tentará executar na CPU.')

## 2. Iniciar Ollama e baixar LFM3-1.7B

Inicia `ollama serve` com `OLLAMA_HOST=0.0.0.0` e `OLLAMA_ORIGINS=*`, como no runner de referência, espera o endpoint local ficar pronto e baixa o modelo. Não abrimos túnel público: o notebook acessa Ollama dentro do próprio runtime.

In [ ]:
import os, time, urllib.request, subprocess
from pathlib import Path

os.environ['OLLAMA_HOST'] = '0.0.0.0'
os.environ['OLLAMA_ORIGINS'] = '*'
ollama_env = os.environ.copy()
ollama_log_path = '/content/ollama.log'
# Evita iniciar um segundo servidor se esta célula for executada novamente.
try:
    with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2):
        ollama_process = None
        print('Servidor Ollama já estava ativo.')
except Exception:
    OLLAMA_LOG = open(ollama_log_path, 'a')
    ollama_process = subprocess.Popen(['ollama', 'serve'], env=ollama_env, stdout=OLLAMA_LOG, stderr=subprocess.STDOUT, start_new_session=True)
for tentativa in range(30):
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2):
            print('Servidor Ollama ativo.')
            break
    except Exception:
        if ollama_process is not None and ollama_process.poll() is not None:
            print(Path(ollama_log_path).read_text(errors='replace')[-4000:])
            raise RuntimeError('Ollama encerrou. Consulte o log exibido acima.')
        time.sleep(1)
else:
    print(Path(ollama_log_path).read_text(errors='replace')[-4000:])
    raise RuntimeError('Ollama não respondeu em 30 segundos. Consulte o log exibido acima.')

subprocess.run(['ollama', 'pull', 'lfm3:1.7b'], check=True)
print('Modelo pronto: lfm3:1.7b')

## 3. Enviar os arquivos do projeto

Selecione `base_enriquecida_tmdb.csv` e `preditor_sucesso.py` (o script sem sklearn). Se estiverem em outro local, monte o Drive e ajuste os caminhos depois.

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecione os dois arquivos
assert Path('/content/base_enriquecida_tmdb.csv').exists(), 'Envie base_enriquecida_tmdb.csv'
assert Path('/content/preditor_sucesso.py').exists(), 'Envie preditor_sucesso.py'
print('Arquivos encontrados e prontos.')

## 4. Treinar e avaliar o algoritmo

A divisão é cronológica. O pipeline aprende normalização e frequências somente no conjunto de treino. Métricas e baseline aparecem abaixo.

In [ ]:
import sys, json, os
sys.path.insert(0, '/content')
from preditor_sucesso import treinar, executar_chat

modelo, preparador, relatorio = treinar('/content/base_enriquecida_tmdb.csv')
print(json.dumps(relatorio, ensure_ascii=False, indent=2))

## 5. Conversar com o preditor

A LLM extrai somente informações ditas pelo usuário. O código mantém os campos, pede o que falta, valida os valores e calcula a estimativa. Digite `sair` para encerrar o chat.

In [ ]:
os.environ['LLM_BASE_URL'] = 'http://127.0.0.1:11434/v1'
os.environ['LLM_API_KEY'] = 'ollama'  # Ollama local não exige chave
os.environ['LLM_MODEL'] = 'lfm3:1.7b'

# Confere a API local novamente: processos de servidor podem parar entre células no Colab.
import time, urllib.request, subprocess
def garantir_ollama_ativo():
    global ollama_process, OLLAMA_LOG
    endpoint = 'http://127.0.0.1:11434/api/tags'
    try:
        with urllib.request.urlopen(endpoint, timeout=3) as resposta:
            return json.loads(resposta.read().decode('utf-8'))
    except Exception:
        pass
    os.environ['OLLAMA_HOST'] = '0.0.0.0'
    os.environ['OLLAMA_ORIGINS'] = '*'
    OLLAMA_LOG = open('/content/ollama.log', 'a')
    ollama_process = subprocess.Popen(['ollama', 'serve'], env=os.environ.copy(), stdout=OLLAMA_LOG, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(45):
        try:
            with urllib.request.urlopen(endpoint, timeout=2) as resposta:
                print('Servidor Ollama iniciado/recuperado.')
                return json.loads(resposta.read().decode('utf-8'))
        except Exception:
            if ollama_process.poll() is not None:
                break
            time.sleep(1)
    print(open('/content/ollama.log', errors='replace').read()[-5000:])
    raise RuntimeError('Ollama não está acessível em 127.0.0.1:11434. O log acima contém a causa do servidor.')

tags = garantir_ollama_ativo()
nomes_modelos = [item.get('name', '') for item in tags.get('models', [])]
if not any(nome == os.environ['LLM_MODEL'] or nome.startswith(os.environ['LLM_MODEL'] + ':') for nome in nomes_modelos):
    subprocess.run(['ollama', 'pull', os.environ['LLM_MODEL']], check=True)

# Testa pelo cliente nativo do Ollama e exige texto não vazio antes do chat.
from ollama import Client
ollama_host = os.environ['LLM_BASE_URL'].removesuffix('/v1')
ollama_client = Client(host=ollama_host)
check = ollama_client.chat(
    model=os.environ['LLM_MODEL'],
    messages=[{'role': 'user', 'content': 'Responda somente com a palavra pronto.'}],
    options={'temperature': 0}, think=False,
)
resposta_teste = check.message.content or ''
print('Resposta do LFM3:', repr(resposta_teste))
if not resposta_teste.strip():
    raise RuntimeError(f'Ollama conectou, mas o modelo retornou texto vazio. thinking={getattr(check.message, "thinking", None)!r}; confira ollama list e o log do servidor.')

executar_chat(modelo, preparador)

## Diagnóstico e observações

- Se Ollama falhar ao iniciar, a célula imprime o final de `/content/ollama.log`. Confira também se selecionou GPU no menu Ambiente de execução → Alterar tipo de ambiente de execução.
- Se o Colab não tiver memória/GPU suficiente, encerre outros processos ou tente uma sessão com mais recursos. O modelo quantizado `lfm3:1.7b` ocupa cerca de 1,1 GB, além da memória usada pelo runtime.
- Para interromper o servidor no fim da sessão: `ollama_process.terminate()`.
- A estimativa não inclui custos de marketing, distribuição ou participação dos cinemas; o score também não é uma probabilidade calibrada.